# Explainability and Interpretation of the Selected CKD Prediction Model

This notebook demonstrates how to use the project's explainability infrastructure end-to-end for the selected chronic kidney disease (CKD) prediction model.

The goal is to interpret the selected model using complementary global and local explanation techniques:

- Feature importance
- Permutation importance
- SHAP values
- LIME local explanations
- Final interpretation for paper inclusion

The analysis identifies model-dependent predictive patterns associated with CKD classification. These explanations must not be interpreted as causal inference.

## 2. Environment and Imports

This section prepares the notebook environment, adds the project `src` directory to the import path, and imports both standard analysis libraries and the project's explainability workflow utilities. The notebook is intended to be run from Jupyter Lab after `uv sync`, without Poetry or `requirements.txt`.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
if Path.cwd() != ROOT:
    os.chdir(ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import FileLink, HTML, Image, Markdown, display
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from explainability.factory import ExplainerFactory
from explainability.reporting import (
    consolidate_feature_rankings,
    save_explainability_report,
)
from explainability.visualizations import plot_pdp_for_top_features
from modeling_workflow import ModelingWorkflow, make_modeling_workflow

RANDOM_STATE = 42
OUTPUT_DIR = Path("reports")
FIGURE_DIR = OUTPUT_DIR / "figures" / "explainability"
TABLE_DIR = OUTPUT_DIR / "tables"

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 120)

## 3. Load the CKD Dataset

The preferred input is the processed CKD dataset because it should contain the cleaned and encoded features used by the modeling notebooks. If no processed CSV is available, the notebook falls back to the interim dataset and validates the target column before continuing.

In [ ]:
def find_prepared_dataset() -> Path:
    """Return the preferred prepared CKD CSV path available in the project."""

    processed_dir = ROOT / "data" / "processed"
    processed_paths = sorted(processed_dir.glob("*.csv")) if processed_dir.exists() else []
    if processed_paths:
        return processed_paths[0]

    interim_path = ROOT / "data" / "interim" / "kidney_disease_interim.csv"
    if interim_path.exists():
        return interim_path

    raise FileNotFoundError(
        "No prepared CKD dataset was found. Expected data/processed/*.csv or "
        "data/interim/kidney_disease_interim.csv."
    )


def infer_target_column(frame: pd.DataFrame) -> str:
    """Infer the CKD target column from common names or ask for manual setup."""

    candidates = ["class", "classification", "target", "ckd", "diagnosis"]
    lower_to_original = {column.lower(): column for column in frame.columns}

    for candidate in candidates:
        if candidate in lower_to_original:
            return lower_to_original[candidate]

    raise ValueError(
        "Could not infer the target column. Set TARGET_COLUMN manually to one of "
        f"the available columns: {list(frame.columns)}"
    )


dataset_path = find_prepared_dataset()
df = pd.read_csv(dataset_path)

index_like_columns = [column for column in df.columns if column.lower().startswith("unnamed")]
if index_like_columns:
    df = df.drop(columns=index_like_columns)

TARGET_COLUMN = infer_target_column(df)

print(f"Loaded dataset: {dataset_path.relative_to(ROOT)}")
print(f"Dataset shape: {df.shape}")
print(f"Target column: {TARGET_COLUMN}")
display(df.head())
display(df[TARGET_COLUMN].value_counts(dropna=False).rename("count").to_frame())

## 4. Prepare X and y

The target column is separated from the predictors, and any remaining categorical variables are encoded with `pandas.get_dummies`. The split uses stratification when possible so that both train and test sets preserve the CKD class distribution.

This section also defines the class that will be explained by SHAP. Text labels such as `ckd` can be inferred directly. If the target is numeric, the notebook requires an explicit mapping before continuing, because `1` should not be assumed to mean CKD.

In [ ]:
X = df.drop(columns=[TARGET_COLUMN]).copy()
y = df[TARGET_COLUMN].copy()

categorical_columns = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
if categorical_columns:
    print(f"Encoding categorical columns with pandas.get_dummies: {categorical_columns}")
    X = pd.get_dummies(X, columns=categorical_columns, drop_first=False)
else:
    print("No object/category predictors were found. Using existing encoded features.")

feature_names = X.columns.tolist()
class_names = [str(value) for value in sorted(pd.Series(y).dropna().unique())]

# For numeric targets only: set this after verifying the dataset documentation.
# Example, only if documented: CKD_POSITIVE_CLASS_VALUE = 1
CKD_POSITIVE_CLASS_VALUE = None


def normalize_label(value) -> str:
    """Normalize labels for robust CKD class detection."""

    return (
        str(value)
        .strip()
        .lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
    )


def infer_positive_class(target) -> object:
    """Infer the CKD class label without assuming numeric label semantics."""

    target_series = pd.Series(target).dropna()
    labels = list(pd.unique(target_series))
    normalized_to_label = {normalize_label(label): label for label in labels}
    ckd_text_labels = [
        "ckd",
        "chronickidneydisease",
        "kidneydisease",
        "disease",
        "positive",
        "yes",
    ]

    for candidate in ckd_text_labels:
        if candidate in normalized_to_label:
            return normalized_to_label[candidate]

    if pd.api.types.is_numeric_dtype(target_series):
        if CKD_POSITIVE_CLASS_VALUE is None:
            raise ValueError(
                "Numeric target labels were detected, but the CKD-positive value "
                "was not documented. Set CKD_POSITIVE_CLASS_VALUE explicitly after "
                "checking the dataset mapping. Do not assume that 1 means CKD."
            )
        if not any(label == CKD_POSITIVE_CLASS_VALUE for label in labels):
            raise ValueError(
                f"CKD_POSITIVE_CLASS_VALUE={CKD_POSITIVE_CLASS_VALUE!r} is not "
                f"present in the target labels: {labels}"
            )
        return CKD_POSITIVE_CLASS_VALUE

    raise ValueError(
        "Could not infer which target label represents CKD. Set or document the "
        "positive class before running explainability. Available labels: "
        f"{labels}"
    )


def build_target_mapping(target, configured_positive_class) -> pd.DataFrame:
    """Document how target labels are interpreted for explainability."""

    rows = []
    for label in sorted(pd.Series(target).dropna().unique()):
        rows.append(
            {
                "target_value": label,
                "interpretation": (
                    "CKD / explained positive class"
                    if label == configured_positive_class
                    else "non-CKD or other class"
                ),
            }
        )
    return pd.DataFrame(rows)


positive_class = infer_positive_class(y)
target_mapping = build_target_mapping(y, positive_class)

stratify_target = y if y.value_counts(dropna=False).min() >= 2 else None
if stratify_target is None:
    print("Stratified split disabled because at least one class has fewer than two samples.")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_target,
)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
display(pd.Series(y_train).value_counts().rename("train_count").to_frame())
display(pd.Series(y_test).value_counts().rename("test_count").to_frame())
print(f"Configured positive_class for explainability: {positive_class!r}")
display(target_mapping)

## 5. Train the Selected Model

Random Forest is used as the default selected model for this demonstration. The model is trained through `make_modeling_workflow`, which wraps the sklearn pipeline and initializes the appropriate explainer through the project's explainability infrastructure.

In [ ]:
classifier = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=1,
)

workflow = make_modeling_workflow(
    classifier=classifier,
    enable_explainability=True,
    feature_names=feature_names,
    class_names=class_names,
    explainability_config={
        "output_dir": OUTPUT_DIR,
        "random_state": RANDOM_STATE,
        # Keep permutation importance sequential on Windows to avoid joblib
        # subprocess import issues when the active .venv has locked DLLs.
        "n_jobs": 1,
        "scoring": "f1_macro",
        "permutation_n_repeats": 10,
        "top_n_features": 15,
        "shap_sample_size": 100,
        "lime_num_features": 10,
        "pdp_top_n": 5,
        "positive_class": positive_class,
    },
)

workflow.fit(X_train, y_train)
y_pred = workflow.predict(X_test)

print(classification_report(y_test, y_pred))
confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=workflow.pipeline.named_steps["classifier"].classes_),
    index=[f"actual_{label}" for label in workflow.pipeline.named_steps["classifier"].classes_],
    columns=[f"predicted_{label}" for label in workflow.pipeline.named_steps["classifier"].classes_],
)
display(confusion)

## 5.1 Explainability Class Configuration

Before generating SHAP and LIME artifacts, the notebook records the classifier classes and the configured `positive_class`. For SHAP plots, positive values increase the model output/support for this configured class; negative values reduce support for this configured class.

In [ ]:
fitted_classifier = workflow.pipeline.named_steps["classifier"]

print("classifier.classes_:", fitted_classifier.classes_)
print("Configured positive_class:", workflow.explainability_config.get("positive_class"))
print("Target mapping used for interpretation:")
display(target_mapping)

if workflow.explainability_config.get("positive_class") not in list(fitted_classifier.classes_):
    raise ValueError(
        "Configured positive_class is not present in classifier.classes_. "
        "Review the target mapping before interpreting SHAP values."
    )

## 6. Global Explainability

This section calls the selected explainer through `workflow.explain_global`.

- Permutation importance measures the degradation of model performance when a feature is shuffled.
- Native feature importance is model-specific and may be biased depending on model structure, feature scale, and feature cardinality.
- SHAP global summaries estimate the average magnitude of feature contributions across evaluated instances.
- Positive SHAP values increase support for the configured `positive_class`; negative SHAP values reduce support for that same configured class.

In [ ]:
global_result = workflow.explain_global(X_test, y_test)

print("Model type:", global_result.model_type)
print("Explained class:", global_result.artifacts.get("explained_class"))
if "shap_global" in global_result.artifacts:
    print(
        "SHAP explained class:",
        global_result.artifacts["shap_global"].get("explained_class"),
    )
print("Warnings:")
for warning in global_result.warnings:
    print(f"- {warning}")

print("Tables:")
display(global_result.tables or {})
print("Figures:")
display(global_result.figures or {})

In [ ]:
if "permutation_importance" in global_result.artifacts:
    display(Markdown("### Permutation Importance"))
    display(global_result.artifacts["permutation_importance"].head(15))

if "native_feature_importance" in global_result.artifacts:
    display(Markdown("### Native Feature Importance"))
    display(global_result.artifacts["native_feature_importance"].head(15))

if "feature_importance" in global_result.artifacts:
    display(Markdown("### Model Feature Importance"))
    display(global_result.artifacts["feature_importance"].head(15))

if "coefficients" in global_result.artifacts:
    display(Markdown("### Linear Coefficients"))
    display(global_result.artifacts["coefficients"].head(15))

shap_global = global_result.artifacts.get("shap_global")
if isinstance(shap_global, dict) and "mean_abs_shap" in shap_global:
    display(Markdown("### Mean Absolute SHAP Values"))
    display(shap_global["mean_abs_shap"].head(15))

## 7. Show Generated Global Visualizations

The explainers save figures through the shared visualization helpers under `reports/figures/explainability`. This section displays those generated files when they are present, keeping plotting logic inside the project infrastructure.

In [ ]:
def display_png_if_exists(path: Path, title: str) -> None:
    """Display a generated PNG figure if it exists."""

    if path.exists():
        display(Markdown(f"### {title}"))
        display(Image(filename=str(path)))
    else:
        print(f"Missing figure: {path}")


global_figures = {
    "Permutation Importance": FIGURE_DIR / "permutation_importance.png",
    "Native Feature Importance": FIGURE_DIR / "native_feature_importance.png",
    "SHAP Global Bar": FIGURE_DIR / "shap_global_bar.png",
    "SHAP Beeswarm": FIGURE_DIR / "shap_beeswarm.png",
}

for title, path in global_figures.items():
    display_png_if_exists(path, title)

## 8. Consolidate Top CKD Features

In [ ]:
top_features = consolidate_feature_rankings(
    global_result,
    top_n=15,
    output_dir=OUTPUT_DIR,
)

display(top_features)

The consolidated ranking combines available signals from permutation importance, model-native importance, coefficients, and SHAP summaries. Features that appear near the top in more than one explanation method receive higher consensus scores.

The highest-ranked features should be described as model-dependent predictors associated with CKD classification. Agreement across methods strengthens the audit signal, but it does not establish causality.

## 9. Local Explainability

Local explanations inspect individual predictions rather than the model as a whole. The selected examples prioritize correctly classified CKD and non-CKD cases, and include a misclassified case when one exists, so the explanation artifacts can support both interpretation and error analysis.

In [ ]:
def find_instance_indices(y_true, y_pred) -> list[int]:
    """Select representative local instances from the test set."""

    y_true_series = pd.Series(y_true).reset_index(drop=True)
    y_pred_series = pd.Series(y_pred).reset_index(drop=True)
    selected: list[int] = []

    class_values = list(y_true_series.dropna().unique())
    ckd_like = [value for value in class_values if "ckd" in str(value).lower()]
    ordered_classes = ckd_like + [value for value in class_values if value not in ckd_like]

    for class_value in ordered_classes:
        matches = y_true_series.eq(class_value) & y_pred_series.eq(class_value)
        if matches.any():
            idx = int(matches[matches].index[0])
            if idx not in selected:
                selected.append(idx)

    misclassified = y_true_series.ne(y_pred_series)
    if misclassified.any():
        idx = int(misclassified[misclassified].index[0])
        if idx not in selected:
            selected.append(idx)

    if not selected:
        selected.append(0)

    return selected[:3]


selected_indices = find_instance_indices(y_test, y_pred)
print(f"Selected local instance positions in X_test: {selected_indices}")

In [ ]:
local_results = []

for idx in selected_indices:
    display(Markdown(f"### Local explanation for test instance position {idx}"))
    result = workflow.explain_local(X_test, idx)
    local_results.append(result)

    print(result.model_type, result.scope)
    print("Explained class:", result.artifacts.get("explained_class"))
    if "shap_local" in result.artifacts:
        print("SHAP explained class:", result.artifacts["shap_local"].get("explained_class"))
    print("Warnings:")
    for warning in result.warnings:
        print(f"- {warning}")

    display(Markdown("#### Prediction Summary"))
    display(result.artifacts["prediction"])

    if "lime" in result.artifacts:
        display(Markdown("#### LIME Local Explanation"))
        display(pd.DataFrame(result.artifacts["lime"]["as_list"], columns=["feature", "weight"]))

    if "neighbors" in result.artifacts:
        display(Markdown("#### KNN Neighbors"))
        display(result.artifacts["neighbors"])

    if "decision_path" in result.artifacts:
        display(Markdown("#### Decision Path"))
        display(result.artifacts["decision_path"])

    if "local_contributions" in result.artifacts:
        display(Markdown("#### Local Contributions"))
        display(result.artifacts["local_contributions"].head(15))

## 10. Show Generated Local Visualizations

Local artifacts vary by explainer type. Tree ensembles can produce SHAP waterfall plots and LIME files, linear models can produce contribution plots, decision trees can export decision paths, and KNN explainers can export neighbor tables.

In [ ]:
def display_local_artifacts(result, idx: int) -> None:
    """Display or link local artifacts generated by explainers."""

    display(Markdown(f"### Local artifacts for test instance position {idx}"))

    for name, path_text in (result.figures or {}).items():
        path = Path(path_text)
        if path.suffix.lower() == ".png" and path.exists():
            display(Markdown(f"#### {name}"))
            display(Image(filename=str(path)))
        elif path.suffix.lower() == ".html" and path.exists():
            display(Markdown(f"#### {name}"))
            display(FileLink(str(path)))

    for name, path_text in (result.tables or {}).items():
        path = Path(path_text)
        if path.exists():
            display(Markdown(f"#### Table: {name}"))
            display(FileLink(str(path)))


for idx, result in zip(selected_indices, local_results):
    display_local_artifacts(result, idx)

## 11. Generate Paper-Ready Report

The reporting helper consolidates the global and local explanation outputs into a Markdown summary. This report is intended as a starting point for the paper's explainability and interpretation section, with warnings and limitations preserved.

In [ ]:
report_path = save_explainability_report(
    model_name=global_result.model_type,
    global_result=global_result,
    local_results=local_results,
    top_features=top_features,
    output_dir=OUTPUT_DIR,
)

print(report_path)
display(Markdown(Path(report_path).read_text(encoding="utf-8")))

## 12. Paper-Ready Interpretation

To interpret the selected CKD prediction model, we combined global and local explainability techniques. Global explanations were obtained using permutation feature importance and model-specific feature importance when available. For tree-based models, SHAP values were computed for the explicitly configured `positive_class` to estimate each feature's contribution to support for that class. Positive SHAP values increase support for the configured class, whereas negative SHAP values reduce support for the configured class. Local explanations were generated for representative test instances using SHAP waterfall plots and LIME. The most relevant predictors were selected based on agreement across complementary explanation methods. These findings should be interpreted as model-dependent predictive patterns rather than causal effects.

In [ ]:
highest_ranked = top_features["feature"].astype(str).head(5).tolist()
method_columns = {
    "permutation importance": "rank_permutation",
    "model-native importance": "rank_native",
    "linear coefficients": "rank_coefficient",
    "SHAP": "rank_shap",
}
identified_methods = [
    method
    for method, column in method_columns.items()
    if column in top_features.columns and top_features[column].notna().any()
]
explained_class = global_result.artifacts.get("explained_class")
if "shap_global" in global_result.artifacts:
    explained_class = global_result.artifacts["shap_global"].get(
        "explained_class",
        explained_class,
    )

paper_ready_notes = f"""
Explained class: {explained_class!r}.

Positive SHAP values indicate increased support for this class. Negative SHAP values indicate reduced support for this class.

The highest-ranked features were: {", ".join(highest_ranked)}.

These features were consistently identified by: {", ".join(identified_methods) if identified_methods else "the available explainability outputs"}.

The generated visualizations are available under `reports/figures/explainability`.
"""

display(Markdown(paper_ready_notes))